## 0. 라이브러리 및 환경 변수 가져오기

In [1]:
import os

from dotenv import load_dotenv
from urllib.parse import quote # 한글 등 인코딩을 위한 quote 함수

load_dotenv()

kakao_api_key = os.getenv("KAKAO_API_KEY")
open_api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_DEFAULT_MODEL")

In [2]:
kakao_api_key

'0554a72192938ec632c8c201c035d16a'

## 1. 웹페이지에 검색 요청하기

In [3]:
import requests
import os

searching = '성수 음식점'

url = 'https://dapi.kakao.com/v2/local/search/keyword.json?query={}'.format(searching)
headers = {
    "Authorization": f"KakaoAK {kakao_api_key}"
}

# url에 접속해 요청을 보내고, 응답을 받음
places = requests.get(url, headers = headers).json() # 받은 응답을 dict로 변환
places

{'documents': [{'address_name': '서울 성동구 성수동1가 656-591',
   'category_group_code': 'FD6',
   'category_group_name': '음식점',
   'category_name': '음식점 > 한식 > 육류,고기 > 갈비',
   'distance': '',
   'id': '12055448',
   'phone': '02-464-3012',
   'place_name': '대성갈비',
   'place_url': 'http://place.map.kakao.com/12055448',
   'road_address_name': '서울 성동구 상원1길 26',
   'x': '127.04843220827088',
   'y': '37.546194169989704'},
  {'address_name': '서울 성동구 성수동1가 656-1021',
   'category_group_code': 'FD6',
   'category_group_name': '음식점',
   'category_name': '음식점 > 양식 > 이탈리안',
   'distance': '',
   'id': '142054376',
   'phone': '02-468-7943',
   'place_name': '누메로도스',
   'place_url': 'http://place.map.kakao.com/142054376',
   'road_address_name': '서울 성동구 상원1길 35-9',
   'x': '127.04736069202619',
   'y': '37.546356784437194'},
  {'address_name': '서울 성동구 성수동1가 656-640',
   'category_group_code': 'CE7',
   'category_group_name': '카페',
   'category_name': '음식점 > 카페 > 테마카페 > 디저트카페',
   'distance': '',
   'i

In [4]:
question = "성수에 있는 레스토랑 두세개 추천해줘."

## 2. GPT 에게 핵심 단어 추출 요청하기

In [5]:
import os
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model = model_name,
    input = [
        {
            "role": "system",
            "content": """ 다음 질문에서 지명 하나와 가장 중요한 키워드 하나를 추출해서 하나의 하나로 붙여줘.
                           원하는 형식:
                           <place>:<keyword>
                    """
        },
        {
            "role": "user",
            "content": question
        }
    ],
    temperature = 0.9, # 1에 가까워질수록 답변의 다양성이 좋아짐
    max_output_tokens=1024, # 답변 최대 길이를 제한
    top_p=1 # 1이면 다양성에 대한 제한 없음을 의미
)
print(response.output_text)

성수:레스토랑


In [6]:
# 답변에서 추출한 항목을 변수에 저장하기
array = response.output_text.split(":")
location = array[0]
keyword = array[1]

location, keyword

('성수', '레스토랑')

## 3. 카카오맵 기반 장소 추천받기

In [7]:
import requests
import json

searching = f"{location} {keyword}"

url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={quote(searching)}"
# url = f"https://dapi.kakao.com/v2/local/search/keyword.json?query={location} {keyword}"

headers = {
    "Authorization": f"KakaoAK {kakao_api_key}"
}

resp = requests.get(url, headers=headers)
resp.raise_for_status() # 요청이 실패하면 오류 발생시키고 멈추기

data = resp.json() # dict 형식으로

items = data.get('documents', [])
context = "\n".join(
    f"{i}) {p.get('place_name','')}, {p.get('address_name','')}, {p.get('category_name','')}"
    for i, p in enumerate(items, 1)
)


print(context)


1) 제스티살룬 성수, 서울 성동구 성수동1가 668-30, 음식점 > 양식 > 햄버거
2) 누메로도스, 서울 성동구 성수동1가 656-1021, 음식점 > 양식 > 이탈리안
3) 이태리국시 성수, 서울 성동구 성수동1가 656-1085, 음식점 > 양식
4) 온량, 서울 성동구 성수동1가 668-54, 음식점 > 양식 > 이탈리안
5) 미테이블 성수본점, 서울 성동구 성수동1가 656-385, 음식점 > 양식
6) 파르코 서울숲점, 서울 성동구 성수동1가 668-134, 음식점 > 양식 > 피자
7) 코너룸, 서울 성동구 성수동1가 13-441, 음식점 > 양식
8) 차만다 성수, 서울 성동구 성수동1가 685-480, 음식점 > 양식
9) 심퍼티쿠시 성수점, 서울 성동구 성수동1가 656-1731, 음식점 > 양식
10) 타코튜즈데이 성수점, 서울 성동구 성수동1가 656-482, 음식점 > 양식 > 멕시칸,브라질
11) 파이프그라운드 서울숲, 서울 성동구 성수동1가 685-700, 음식점 > 양식
12) 이태리차차, 서울 성동구 성수동1가 16-25, 음식점 > 양식 > 이탈리안
13) 바스버거 성수점, 서울 성동구 성수동1가 656-819, 음식점 > 양식 > 햄버거 > 바스버거
14) 갓잇 서울숲점, 서울 성동구 성수동1가 656-402, 음식점 > 양식 > 멕시칸,브라질
15) 누메로뜨레쓰, 서울 성동구 성수동1가 656-1661, 음식점 > 양식 > 이탈리안


### 총정리. GPT 에게 검색 결과를 보고 질문에 답하도록 요청하기

In [9]:
%%writefile search_places.py

import os
import json
import requests
from detenv import load_dotenv
from openai import OpenAI

# .env 파일에서 API 키와 모델명 불러오기
load_dotenv()

kakao_api_key = os.getenv("KAKAO_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_DEFAULT_MODEL")


# OpenAI 클라이언트 생성
client = OpenAI(api_key=open_api_key)


# 사용자 질문에서 장소 검색에 필요한 지역명과 키워드 추출
def extract_search_terms(question):
    response client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": """
                다음 질문에서 장소 검색에 필요한 지역명과 키워드를 추출하라.
                반드시 JSON 형식으로만 답하라.
                형식:
                {
                 "location": "지역명",
                 "keyword": "장소 키워드"                    
                }"""
            },
            {
                "role": "user",
                "content": "question"
            }
        ],
        temperature=0,
        max_output_tokens=300,
        top_p=1
    )

    return josn.loads(response.output_text)


# 카카오 장소 검색 API로 실제 장소 목록 조회
def search_kakao_places(location, keyword):
    searching = f"{location} {keyword}"

    url = "https://dapi.kakao.com/v2/local/search/keyword.json"

    # 카카오 REST API 키를 헤더에 포함
    headers = {
        "Authorization": f"KakaoAK {kakao_api_key}"
    }

    # 검색어와 검색 개수 설정
    params = {
        "query": searching,
        "size": 5
    }

    # 카카오 API 요청
    resp = requests.get(url, headers=headers, params=params)
    resp.raise_for_status()

    # 검색 결과 중 장소 목록만 반환
    data = resp.json()
    return data.get("documents", [])


# 검색된 장소 목륵을 ChatGPT가 읽기 쉬운 텍스트로 변환
def make_context(places):
    return "\n".join(
        f"{i}) {p.get('place_name', '')}, "
        f"{p.get('address_name', '')}, "
        f"{p.get('category', '')}, "
        f"{p.get('place_url', '')}"
        for i, p in enumerate(places, 1)
    )


# 검색 결과를 근거로 사용자 질문에 대한 최종 답변 생성
def answer_user_question(question, context):
    response = client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "context": """당신은 친절한 여행 가이드입니다.
                        아래 검색 결과를 참고하여 사용자의 질문에 답변하세요.
                        """
            },
            {
                "role": "user",
                "context": f"검색 결과:\n{context}\n\n질문: {question}"
            }
        ],
        temperature=0.9,
        max_output_tokens=1024,
        top_1=1
    )

    return response.output_text

# 사용자 질문 입력
question = "성수에 있는 레스토랑 두세개 추천해줘."

# 1. 질문에서 검색어 추출
terms = extract_search_terms(question)

# 2. 카카오 API로 장소 검색
places = search_kakao_places(terms["location"], terms["keyword"])

# 3. 검색 결과를 LLM 입력용 텍스트로 변환
context = make_context(places)

# 4. 검색 결과 기반으로 최종 답변 생성
answer = answer_user_question(question, context)

# 5. 답변 출력
print(answer)



Writing search_places.py
